# 01 — Exploration

Exploratory look at the feature panel. **The pipeline itself lives in `src/momentum_ml/`**; this notebook only reads its outputs.

Run `python -m momentum_ml.run_pipeline --config configs/sample.yaml` (or the real-data config) first.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from momentum_ml.config import load_config
from momentum_ml.build_features import load_panel
from momentum_ml.features import feature_columns

CONFIG = 'configs/sample.yaml'  # switch to configs/config.yaml for real data
cfg = load_config(CONFIG)
panel = load_panel(cfg)
feats = feature_columns(panel)
panel.head()

## Base rates by ticker and year

In [ ]:
panel.assign(year=panel['date'].dt.year).pivot_table(index='year', columns='ticker', values='target', aggfunc='mean').round(3)

## Rank correlation of each feature with the next-day return

Spearman correlations by ticker. Values near zero are the norm; treat anything else with suspicion (multiple comparisons!).

In [ ]:
ic = panel.groupby('ticker').apply(lambda d: d[feats].corrwith(d['fwd_ret'], method='spearman'))
ic.T.style.background_gradient(cmap='RdBu', vmin=-0.1, vmax=0.1).format('{:.3f}')

## Autocorrelation of daily returns

In [ ]:
ac = panel.groupby('ticker')['ret_1d'].apply(lambda s: pd.Series({f'lag{k}': s.autocorr(k) for k in range(1, 6)}))
ac.unstack().round(3)

## Feature correlation (redundancy check)

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(panel[feats].corr(), cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature correlation matrix')
plt.show()